In [1]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
import gradio as gr
from langchain_core.messages import HumanMessage, SystemMessage

In [2]:
MODEL = "gpt-4.1-nano"
DB_NAME = "vector_db"
load_dotenv(override=True)

True

In [5]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vector_store = Chroma(
    embedding_function=embeddings,
    persist_directory=DB_NAME
)



In [6]:
retriever = vector_store.as_retriever()

llm = ChatOpenAI(model=MODEL, temperature=0)

In [20]:
def generate_system_prompt(query):
    SYSTEM_PROMPT_TEMPLATE =f"""
    You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{query}
"""
    return SYSTEM_PROMPT_TEMPLATE

In [23]:
def generate_response(question:str,history:list[str]):
    docs = retriever.invoke(question)
    context ="\n\n".join(doc.page_content for doc in docs)
    
    sys_prompt = generate_system_prompt(context)
    # print(sys_prompt)
    response = llm.invoke([SystemMessage(content=sys_prompt),HumanMessage(content=question)])
    return response.content


In [26]:
print(retriever.invoke("Hi"))

[Document(id='70eb8b98-eade-476f-a76f-999e5e97ac02', metadata={'source': 'knowledge-base/contracts/Contract with Heritage Life Assurance for Lifellm.md', 'doc_type': 'contracts'}, page_content='6. **Advanced Reporting**:\n   - Application conversion analytics\n   - Underwriting productivity metrics\n   - Policy persistency reporting\n   - Agent performance dashboards\n\n## Support\n\n1. **Technical Support**: Heritage will receive priority support:\n   - Monday-Friday 7 AM - 7 PM EST\n   - Email, phone, and chat support\n   - 8-hour response time for critical issues\n   - Online knowledge base\n\n2. **Training**: Training program includes:\n   - 3-week implementation\n   - Training for up to 15 staff members (25 hours)\n   - Quarterly webinars on platform updates\n   - Online training library access\n\n3. **Updates**: Monthly platform enhancements and security patches. Maintenance windows: Sunday 1 AM - 5 AM EST.\n\n4. **Account Management**: Named customer success manager with quarter

In [24]:
print(generate_response("What does she do?",[]))

Could you please specify which person you're referring to? I have information on several individuals, such as Samantha Greene, Emily, and Maxine. Let me know which one you'd like to learn more about!


In [25]:
gr.ChatInterface(generate_response,title="Insurellm",description="Ask anything about Insurellm").launch(inbrowser=True)

/Users/siddarthalegala/projects/LLMEngineeringLearning/.venv/lib/python3.12/site-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
